In [ ]:
import os
import pandas as pd
import numpy as np
import cv2
from typing import List, Dict, Tuple, Optional, Final

In [ ]:
EXP_PATH: str = './nextflow_outputs/20231013_Alpha007_R000242_A123456_Cell-interaction_2lanes_20231013_NK_K562_48h_RY'
SCANS_LIST: List[int] = [3, 4, 5, 6, 7, 8, 9, 10, 11]
LANES_LIST: List[int] = [7, 8]
GATED_CLASSES_PATH = os.path.join(EXP_PATH, 'gated_classes')
OUTPUT_FOLDER = 'live_dead_cell_classes'
if not os.path.exists(OUTPUT_FOLDER):
    os.mkdir(OUTPUT_FOLDER)

CLASSES = ['live_cell', 'dead_cell']

for class_name in CLASSES:
    if not os.path.exists(os.path.join(OUTPUT_FOLDER, class_name)):
        os.mkdir(os.path.join(OUTPUT_FOLDER, class_name))

SKIP_CELLS_CLOSE_TO_CAGE_BOUNDARY: bool = True
# a folder for storing images of cells close to the cage boundary; only used for debugging and making sure we are filtering
# those cells
# if not os.path.exists(os.path.join(OUTPUT_FOLDER, 'cells_close_to_cages')):
#     os.mkdir(os.path.join(OUTPUT_FOLDER, 'cells_close_to_cages'))

In [ ]:
gated_files = os.listdir(GATED_CLASSES_PATH)
live_cells_df: Dict[Tuple[int, int], pd.DataFrame] = {}
live_cells_idxs: Dict[Tuple[int, int], np.ndarray] = {}
    
dead_cells_df: Dict[Tuple[int, int], pd.DataFrame] = {}
dead_cells_idxs: Dict[Tuple[int, int], np.ndarray] = {}
    
for file in gated_files:
    print(file)
    scan_num, lane_num = [int(s) for s in file[25:].strip().split('_')[:2]]
    if 'live' in '_'.join(file[25:].strip().split('_')[2:]).lower():
        # live cell annotations
        live_cells_df[(scan_num, lane_num)] = pd.read_csv(os.path.join(GATED_CLASSES_PATH,  file), sep="\t")
        live_cells_idxs[(scan_num, lane_num)] = live_cells_df[(scan_num, lane_num)]['eventNumber'].values
    elif 'dead' in '_'.join(file[25:].strip().split('_')[2:]).lower():
        # dead cell annotations
        dead_cells_df[(scan_num, lane_num)] = pd.read_csv(os.path.join(GATED_CLASSES_PATH,  file), sep="\t")
        dead_cells_idxs[(scan_num, lane_num)] = dead_cells_df[(scan_num, lane_num)]['eventNumber'].values
    else:
        print(f"[WARN] Unable to extract the class (dead/live cell) from summary file {file}")
        
for scan_num in SCANS_LIST:
    for lane_num in LANES_LIST:
        if (scan_num, lane_num) not in dead_cells_df or (scan_num, lane_num) not in live_cells_df:
            print(f"[WARN] A gated summary file is missing for scan {scan_num}, lane {lane_num}")

In [ ]:
master_csv: Dict[Tuple[int, int], pd.DataFrame] = {}
all_cell_areas = []
for scan_num in SCANS_LIST:
    for lane_num in LANES_LIST:
        
        csv_df = pd.read_csv(os.path.join(EXP_PATH, "scan_" + str(scan_num), "lane_" + str(lane_num), 
                                          "processed/FEATURE_EXTRACTION/primary_analysis_summary.csv")) 
        
        master_csv[(scan_num, lane_num)] = csv_df
        # extract the cell crop around the cell center (specified by "object_x_crop" and "object_y_crop")
        # in the cage crop specified by "file_name" field in master_csv, for each class
        # before doing that, let's pick a proper crop size
        # the crop should be large enough to cover the whole cell, but should not be very large to include any adjacent cell
        # (should be unlikely as the cells are caged)
        cell_areas: np.ndarray = csv_df.loc[csv_df["is_cell"] == 1, "object_area_px"].values
        print(f"99th percentile of cell areas ({scan_num}, {lane_num}): {np.percentile(cell_areas, 99)}")
        print(f"Maximum of cell areas ({scan_num}, {lane_num}): {cell_areas.max()}")
        
        all_cell_areas += list(cell_areas)

print(f"99th percentile of cell areas over all scans/lanes: {np.percentile(all_cell_areas, 99)}")
print(f"Maximum of cell areas over all scans/lanes:: {np.max(all_cell_areas)}")
          

In [ ]:
print(int(np.sqrt(8000 / np.pi)))
# a crop of size 108x108 pixels around the center seems reasonable

# Note that there are multiple cells in one cage potentially in close proximity of each other
# in this experiment
# we want to use a radius as small as possible to have only on cell in an image
CROP_HALF_SIZE: int = 48

## Create cropped cell images of different classes
Skip this step onces the image folders are created. This step should be repeated for any updated run of the pipeline (more results), or any change in the cell gating.

In [ ]:
num_dead_cell_samples: int = 0
num_live_cell_samples: int = 0

for scan_num in SCANS_LIST:
    for lane_num in LANES_LIST:
        
        print(f"Processing scan {scan_num}, lane {lane_num} ...")
        
        for idx, row in master_csv[(scan_num, lane_num)].iterrows():

            cage_crop_file: str = row['file_name']
            # it is possible to have multiple "object"s (cells) from the same cage crop 
            # (mutiple cells in one cage), use the object_id to save them into different files
            object_id: int = row['object_id']
            cell_center_x: int = row['object_x_crop']
            cell_center_y: int = row['object_y_crop']
            img: np.ndarray = cv2.imread(os.path.join(EXP_PATH, "FC_NAVIGATOR", "scan_" + str(scan_num), 
                                                      "lane_" + str(lane_num), "CAGE_CROPS/CROPS_White", 
                                                      cage_crop_file + "_White.png"), cv2.IMREAD_UNCHANGED)
    
            crop_height, crop_width = img.shape[:2]
            num_channels: int = 1
            if len(img.shape) > 2:
                num_channels = img.shape[2]

            start_pixel_in_x: int = max(0, cell_center_x - CROP_HALF_SIZE)
            end_pixel_in_x: int = min(crop_width, cell_center_x + CROP_HALF_SIZE)
    
            start_pixel_in_y: int = max(0, cell_center_y - CROP_HALF_SIZE)
            end_pixel_in_y: int = min(crop_height, cell_center_y + CROP_HALF_SIZE)

            # make sure the saved image is always of size (2 * CROP_HALF_SIZE, 2 * CROP_HALF_SIZE) (may not be true for cells close to the
            # boundary of the image)
            # if not, we zero pad on the right and bottom
            if num_channels > 1:
                cell_crop: np.ndarray = np.zeros((2 * CROP_HALF_SIZE, 2 * CROP_HALF_SIZE, num_channels), np.uint8)
            else:
                cell_crop: np.ndarray = np.zeros((2 * CROP_HALF_SIZE, 2 * CROP_HALF_SIZE), np.uint8)

            cell_crop[:end_pixel_in_y - start_pixel_in_y , :end_pixel_in_x - start_pixel_in_x] = \
            img[start_pixel_in_y: end_pixel_in_y, start_pixel_in_x: end_pixel_in_x]

            # identify cells close to the cage boundary, we are going to exclude them
            distance_to_cage_center: float = row['cage_center_dist']
            cage_radius: float = row['cage_radius']

            if SKIP_CELLS_CLOSE_TO_CAGE_BOUNDARY and \
            np.abs(cage_radius - distance_to_cage_center) <= 2 * CROP_HALF_SIZE:
                # cv2.imwrite(os.path.join(OUTPUT_FOLDER, 'cells_close_to_cages', cage_crop_file + '_' + str(object_id) + '.jpg'), 
                #             cell_crop, [int(cv2.IMWRITE_JPEG_QUALITY), 100])
                continue
    
            # we are going to ignore rows with 'is_cell' == 0 (no bg class is needed)
            if row['is_cell'] == 0:
                continue
            
            if idx in live_cells_idxs[(scan_num, lane_num)]:
                num_live_cell_samples += 1 
                cv2.imwrite(os.path.join(OUTPUT_FOLDER, 'live_cell', cage_crop_file + '_' + str(object_id) + '.jpg'), 
                            cell_crop, [int(cv2.IMWRITE_JPEG_QUALITY), 100])
            elif idx in dead_cells_idxs[(scan_num, lane_num)]:
                num_dead_cell_samples += 1
                cv2.imwrite(os.path.join(OUTPUT_FOLDER, 'dead_cell', cage_crop_file + '_' + str(object_id) + '.jpg'), 
                            cell_crop, [int(cv2.IMWRITE_JPEG_QUALITY), 100])

In [ ]:
num_dead_cell_samples

In [ ]:
num_live_cell_samples

# Training a Classifier

In [ ]:
import torch, torchvision
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import time

import random
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from PIL import Image

## Configurations

In [ ]:
# percentage of images to be assigned to test set
# per each class/label
TEST_TO_TRAIN_RATIO = 0.10

# random seed for dividing the images to train and test
# set
SEED = 7

# training batch size
BATCH_SIZE = 32
# learning rate
LEARNING_RATE = 1e-3
# number of training epochs
NUM_EPOCHS = 18
# learning rate decay steps
LR_DECAY_STEPS = 0

## Dataset model

In [ ]:
class BloodCellDataset(Dataset):
    """ Dataset of blood cells of different types (classes) """

    def __init__(self, class_images_dict: Dict[int, List[str]], transform=None):
        """
        Args:
            class_images_dict (dictionary): A dictionary with keys as class 
                IDs (integers 0 to num_classes; 0 reserved for background if exists) and 
                values as the full path to the list of train/test images for the class ID 
                (the name should include the full path to the image).
            transform (callable, optional): Optional transform to be applied
                on a sample
        """
        
        self.image_names: List[str] = []
        self.labels: List[int] = []
        for label, image_paths_list in class_images_dict.items():
            self.image_names = self.image_names + image_paths_list
            self.labels = self.labels + [label] * len(image_paths_list)
        self.transform = transform
        self.num_classes = len(class_images_dict)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        
        image: np.ndarray = cv2.imread(self.image_names[idx], cv2.IMREAD_UNCHANGED)
        
        if np.ndim(image) == 2:
            # for gray scale channel, make them a 3-D image expected by the model
            image = np.repeat(np.expand_dims(image, axis=2), 3, axis=2)
        
        # image_tensor: torch.tensor = torchvision.transforms.functional.to_tensor(image)
        
        if self.transform:
            image = self.transform(image)

        
        return image, self.labels[idx]

## Data transform

In [ ]:
# train and test data transforms
image_transforms = { 
    'train': torchvision.transforms.Compose([
        torchvision.transforms.ToTensor(),
        torchvision.transforms.RandomApply(torch.nn.ModuleList([
            torchvision.transforms.RandomRotation(degrees=[90.0, 90.0])
        ]), p=0.25),
        torchvision.transforms.RandomHorizontalFlip(p=0.5),
        torchvision.transforms.RandomVerticalFlip(p=0.5),
        torchvision.transforms.Normalize([0.485, 0.456, 0.406],
                                         [0.229, 0.224, 0.225])
    ]),
    'test': torchvision.transforms.Compose([
        torchvision.transforms.ToTensor(),
        torchvision.transforms.Normalize([0.485, 0.456, 0.406],
                                         [0.229, 0.224, 0.225])
    ])
}

In [ ]:
random.seed(SEED)
class_images_dict_test: Dict[int, List[str]] = {}
class_images_dict_train: Dict[int, List[str]] = {}
label_map: Dict[int, str] = {}
for class_id, class_name in enumerate(CLASSES):
    image_files = os.listdir(os.path.join(OUTPUT_FOLDER, class_name))
    # shuffle them before assigning the images to test and train
    random.shuffle(image_files)
    num_test_images: int = int(TEST_TO_TRAIN_RATIO * len(image_files))
    class_images_dict_test[class_id] = [os.path.join(OUTPUT_FOLDER, class_name, f) for f in image_files[:num_test_images]]
    class_images_dict_train[class_id] = [os.path.join(OUTPUT_FOLDER, class_name, f) for f in image_files[num_test_images:]]
    label_map[class_id] = class_name
    print(f"[INFO] Found {len(image_files)} images of class '{class_name}' with ID {class_id}")
    print(f"[INFO] splitted into {len(class_images_dict_test[class_id])}/{len(class_images_dict_train[class_id])} for test/train sets ")

## Datasets and dataloaders

In [ ]:
train_dataset = BloodCellDataset(class_images_dict_train, image_transforms['train'])
test_dataset = BloodCellDataset(class_images_dict_test, image_transforms['test'])

train_data_loader = DataLoader(train_dataset, batch_size = BATCH_SIZE, shuffle = True)
test_data_loader = DataLoader(test_dataset, batch_size = 1, shuffle = False)

### Visualize some samples

In [ ]:
def show_sample_batch(sample_batch):
    """ Show training images for a batch """
    images_batch, labels_batch = sample_batch
    batch_size: int = len(labels_batch)
    grid_image = torchvision.utils.make_grid(images_batch, normalize = True)
    plt.imshow(grid_image.numpy().transpose((1, 2, 0)))
    print('Labels:' + ' '.join('%5s' % labels_batch[j].item() for j in range(batch_size)))

In [ ]:
data_iter = iter(DataLoader(train_dataset, batch_size = 4, shuffle = True))
show_sample_batch(next(data_iter))     

In [ ]:
deads = []
lives = []
for i in range(len(train_dataset)):
    img_tensor, l = train_dataset[i]
    if l == 0:
        if len(lives) >= 256:
            continue
        lives.append(img_tensor)
    else:
        if len(deads) >= 256:
            continue
        deads.append(img_tensor)

In [ ]:
len(lives)

In [ ]:
len(deads)

In [ ]:
images_batch = torch.cat([l.unsqueeze(0) for l in deads], dim=0)
grid_image = torchvision.utils.make_grid(images_batch, normalize = True, nrow=16)
img = grid_image.mul(255).byte().permute((1, 2, 0)).numpy()
display(Image.fromarray(img))

In [ ]:
images_batch = torch.cat([l.unsqueeze(0) for l in lives], dim=0)
grid_image = torchvision.utils.make_grid(images_batch, normalize = True, nrow=16)
img = grid_image.mul(255).byte().permute((1, 2, 0)).numpy()
display(Image.fromarray(img))

## Model definition
We use a ResNet50 for the classifier, after replacing the head.

In [ ]:
def build_model(num_classes: int, freeze_backbone: bool = False):
    # get the pre-trained ResNet50 model
    model = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.DEFAULT)
    if freeze_backbone:
        # freeze all model parameters (except the FC layer that will be replaced)
        for param in model.parameters():
            param.requires_grad_(False)
    
    # change/replace the final layer of ResNet50 model for Transfer Learning
    num_fc_inputs: int = model.fc.in_features

    model.fc = nn.Sequential(
        nn.Linear(num_fc_inputs, 256),
        nn.ReLU(),
        nn.Dropout(0.4),
        nn.Linear(256, num_classes), # nuumber of output classes
        )
    return model


## Training
### Optimizer settings

In [ ]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print('Device available:' , device)

resnet_50_model = build_model(len(label_map))

# move model to the right device
resnet_50_model.train()
resnet_50_model.to(device)

# construct an optimizer
params = [p for p in resnet_50_model.parameters() if p.requires_grad]
optimizer = torch.optim.Adam(params, lr = LEARNING_RATE)
print('Adam Optimizer is configured for %d epochs' %NUM_EPOCHS)

print(f"Initial learning rate is set to {LEARNING_RATE}")
if LR_DECAY_STEPS < 1:
    print(f"One-Cyle LR scheduler is configured for {NUM_EPOCHS} with {len(train_data_loader)} steps per epoch")
    lr_scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer=optimizer, 
                                                       max_lr=LEARNING_RATE, 
                                                       epochs=NUM_EPOCHS,
                                                       steps_per_epoch=len(train_data_loader))
    
else:
    print(f"Step LR scheduler is configured with {LR_DECAY_STEPS} epochs for each step")
    lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer=optimizer,
                                                   step_size=LR_DECAY_STEPS,
                                                   gamma=0.1)


criterion = nn.CrossEntropyLoss()

### Training script

In [ ]:
def train(model, 
          train_loader, 
          test_loader, 
          criterion, 
          optimizer, 
          lr_scheduler,
          num_epochs,
          device):
    
    torch.cuda.empty_cache()
    
    # losses, accuracies and mean IoUs over the training epochs
    train_losses: List[float] = []
    test_losses: List[float] = []
    train_accs: List[float] = []
    test_accs: List[float] = []
    test_precision: List[float] = []
    test_recall: List[float] = []
    
    # learning rates used for each step (not each epoch as we may use OneCyle scheduling)
    lrs: List[float] = []
    min_loss: float = np.inf
    
    model = model.to(device)
    
    start_time = time.time()
    
    for epoch in range(num_epochs):
        
        since = time.time()
        
        running_loss: float = 0
        accuracy: float = 0
        
        # training loop
        model.train()
        for i, data in enumerate(tqdm(train_loader)):
            # training phase
            image_tensors, label_tensors = data
            
            image_tensors = image_tensors.to(device) 
            label_tensors = label_tensors.to(device)
            
            # forward
            output = model(image_tensors)
            loss = criterion(output, label_tensors)
            # evaluate metrics
            # compute the accuracy
            
            _, predictions = torch.max(output.data, dim=1)
            correct = predictions.eq(label_tensors.data.view_as(predictions)).int()
            
            # convert correct_counts to float and then compute the mean
            accuracy += float(correct.sum()) / float(correct.numel())
            
            # backward
            loss.backward()
            optimizer.step() # update weight          
            optimizer.zero_grad() # reset gradient
            
            # update the learning rate only after one batch in case of One-Cycle LR scheduler
            lrs.append(lr_scheduler.get_last_lr()[0])
            if isinstance(lr_scheduler, torch.optim.lr_scheduler.OneCycleLR):
                lr_scheduler.step() 
            
            running_loss += loss.item()
        
        # update the learning rate after one full epoch if LR step scheduler is used
        if isinstance(lr_scheduler, torch.optim.lr_scheduler.StepLR):
            lr_scheduler.step() 
        
        # run the validation after each training epoch
        model.eval()
        test_running_loss: float = 0
        test_accuracy: float = 0
        
        # validation loop
        with torch.no_grad():
            for i, data in enumerate(tqdm(test_loader)):
                
                image_tensors, label_tensors = data
            
                image_tensors = image_tensors.to(device) 
                label_tensors = label_tensors.to(device)

                output = model(image_tensors)
                # evaluation metrics
                # compute the accuracy
                
                _, predictions = torch.max(output.data, dim=1)
                correct = predictions.eq(label_tensors.data.view_as(predictions)).int()
                
                # convert correct_counts to float and then compute the mean
                test_accuracy += float(correct.sum()) / float(correct.numel())
                # loss
                loss = criterion(output, label_tensors)                                  
                test_running_loss += loss.item()
            
        # calculatio mean for each batch
        running_loss /= len(train_loader)
        accuracy /= len(train_loader)
        
        test_running_loss /= len(test_loader)
        test_accuracy /= len(test_loader)
                       
         # save the results
        train_losses.append(running_loss)
        train_accs.append(accuracy)
        
        test_losses.append(test_running_loss)
        test_accs.append(test_accuracy)
        
        print('saving the model ...')
        torch.save(model.state_dict(), os.path.join('classifier_models', 'checkpoint_' + str(epoch) +'.pt'))
                    
        
        print("Epoch:{}/{} ... \n".format(epoch + 1, num_epochs),
              "Train Loss: {:.3f} \n".format(running_loss),
              "Test Loss: {:.3f} \n".format(test_running_loss),
              "Train Accuracy: {:.3f} \n".format(accuracy),
              "Test Accuracy: {:.3f} \n".format(test_accuracy),
              "Time: {:.2f} m".format((time.time() - since) / 60))
        
    history = {'train_loss' : train_losses, 'test_loss': test_losses,
               'train_acc': train_accs, 'val_acc': test_accs,
               'lrs': lrs}
    print('Total time: {:.2f} m' .format((time.time()- start_time) / 60))
    return history

In [ ]:
history = train(resnet_50_model, 
          train_data_loader, 
          test_data_loader, 
          criterion, 
          optimizer, 
          lr_scheduler,
          NUM_EPOCHS,
          device)

## Evaluation
### Confusion matrix

In [ ]:
# load the best model_dict from file after training 

resnet_50_model = build_model(len(CLASSES))
resnet_50_model.load_state_dict(torch.load(os.path.join('classifier_models', 'best_2.pt')))
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
resnet_50_model.to(device)
resnet_50_model.eval()

In [ ]:
def predict(model, image_tensor):
    image_tensor: torch.tensor = image_tensor.unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():   
        # model outputs log probabilities
        out = model(image_tensor)
        # total = torch.exp(out).sum().item()
        # topk, topclass = out.topk(3, dim=1)
        # for i in range(3):
        #     print("Predcition", i + 1, ":", topclass.cpu().numpy()[0][i], ", Score: ", np.exp(topk.cpu().numpy()[0][i])/total)
        ret, prediction = torch.max(out.data, 1)
        return prediction.item()

In [ ]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score 
import seaborn as sn

y_pred = []
y_true = []

# iterate over test data
for image_tensors, label_tensors in test_data_loader:
    image_tensors = image_tensors.to(device)
    label_tensors = label_tensors.to(device)    
    outputs = resnet_50_model(image_tensors) # Feed Network

    predictions = (torch.max(torch.exp(outputs), 1)[1]).data.cpu().numpy()
    y_pred.extend(predictions) # Save Prediction
        
    labels = label_tensors.data.cpu().numpy()
    y_true.extend(labels) 

# Build confusion matrix
cf_matrix = confusion_matrix(y_true, y_pred)
df_cm = pd.DataFrame(cf_matrix / np.sum(cf_matrix, axis=1)[:, None], index = [i for i in CLASSES],
                     columns = [i for i in CLASSES])
plt.figure(figsize = (12,7))
sn.heatmap(df_cm, annot=True)
print(f"Accuracy: {np.round(accuracy_score(y_pred, y_true) * 100, 2)}%")

## Extract emebeddings of the train set

In [ ]:
# replace the fully connected layer with an identity layer
resnet_50_model.fc = nn.Identity()
resnet_50_model.to(device)
resnet_50_model.eval()

In [ ]:
embeddings = []
labels = []
for data in train_data_loader:
    image_tensors, label_tensors = data
    image_tensors = image_tensors.to(device)
    label_tensors = label_tensors.to(device)    
   
    with torch.no_grad():
         features = resnet_50_model(image_tensors)
    embeddings.append(features)
    labels.append(label_tensors)
embeddings = torch.cat(embeddings, dim = 0)
labels = torch.cat(labels, dim = 0)

In [ ]:
# convert to numpy before any dimentionality reduction
embeddings = embeddings.detach().cpu().numpy()
labels = labels.detach().cpu().numpy()

## T-SNE visualization

In [ ]:
from sklearn.manifold import TSNE
# we want to get T-SNE embedding with 2 dimensions
n_components = 2
tsne = TSNE(n_components)
tsne_result = tsne.fit_transform(embeddings)
# Plot the result of our TSNE with the label color coded
# A lot of the stuff here is about making the plot look pretty and not TSNE
tsne_result_df = pd.DataFrame({'tsne_1': tsne_result[:,0], 'tsne_2': tsne_result[:,1], 'label': labels})
tsne_result_df['label'] = tsne_result_df['label'].map(label_map)
fig, ax = plt.subplots(1)
sn.scatterplot(x='tsne_1', y='tsne_2', hue='label', data=tsne_result_df, ax=ax,s=120)
lim = (tsne_result.min()-5, tsne_result.max()+5)
ax.set_xlim(lim)
ax.set_ylim(lim)
ax.set_aspect('equal')
ax.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.0)